In [1]:
!pip install pandas scikit-learn matplotlib sympy --target=/venv/main/lib/python3.12/site-packages

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.5/117.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 38.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 63.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 66.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 66.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 362.6/362.6 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 68.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.1/309.1 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 54.5 MB/s eta 0:00:00
   ━━━━━━━━━━━

In [2]:
# Cell 1: Setup + H100 GPU Check
import re, time, random, pickle, urllib.request
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import Counter

import sympy as sp
from sympy import (symbols, series, sin, cos, exp, log, tan, sqrt,
                   atan, asin, acos, sinh, cosh, tanh, Rational,
                   expand, zoo, nan, oo, I, Symbol)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

SEED = 42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
x = Symbol('x')
print("✅ Ready!")

Device: cuda
GPU: NVIDIA H100 NVL
VRAM: 99.9 GB
✅ Ready!


In [3]:
# Cell 2: Load 60K data (skip SymPy generation)
print("="*60)
print("LOADING PROSPER'S 57K DATASET")
print("="*60)

# Tokenizer
class MathTokenizer:
    PAD_TOKEN='<PAD>'; SOS_TOKEN='<SOS>'; EOS_TOKEN='<EOS>'; UNK_TOKEN='<UNK>'
    def __init__(self):
        self.token2idx={}; self.idx2token={}; self.vocab_size=0
        for i,t in enumerate([self.PAD_TOKEN,self.SOS_TOKEN,self.EOS_TOKEN,self.UNK_TOKEN]):
            self.token2idx[t]=i; self.idx2token[i]=t
        self.vocab_size=4
    def tokenize(self, s):
        s=s.replace(" ","")
        raw=re.findall(r'(sin|cos|tan|exp|log|sqrt|sinh|cosh|tanh|asin|acos|atan|\*\*|\d+|\+|\-|\*|/|\(|\)|x)',s)
        tokens=[]
        for t in raw:
            if t.isdigit() and len(t)>1:
                for d in t: tokens.append(d)
            else: tokens.append(t)
        return tokens
    def build_vocab(self, exprs=None):
        for t in ['0','1','2','3','4','5','6','7','8','9','+','-','*','/','**','(',')','x',
                   'sin','cos','tan','exp','log','sqrt','sinh','cosh','tanh','asin','acos','atan']:
            if t not in self.token2idx: self.token2idx[t]=self.vocab_size; self.idx2token[self.vocab_size]=t; self.vocab_size+=1
        if exprs:
            for e in exprs:
                for t in self.tokenize(e):
                    if t not in self.token2idx: self.token2idx[t]=self.vocab_size; self.idx2token[self.vocab_size]=t; self.vocab_size+=1
        print(f"Vocabulary: {self.vocab_size} tokens"); return self
    def encode(self, s, add_sos=False, add_eos=False):
        ids=[]
        if add_sos: ids.append(self.token2idx[self.SOS_TOKEN])
        for t in self.tokenize(s): ids.append(self.token2idx.get(t, self.token2idx[self.UNK_TOKEN]))
        if add_eos: ids.append(self.token2idx[self.EOS_TOKEN])
        return ids
    def decode(self, ids):
        tokens=[self.idx2token.get(i,self.UNK_TOKEN) for i in ids if self.idx2token.get(i,'') not in [self.PAD_TOKEN,self.SOS_TOKEN,self.EOS_TOKEN]]
        r=''; i=0
        while i<len(tokens):
            t=tokens[i]
            if t.isdigit():
                n=t
                while i+1<len(tokens) and tokens[i+1].isdigit(): i+=1; n+=tokens[i]
                r+=n
            elif t in ['+','-']: r+=' '+t+' '
            else: r+=t
            i+=1
        return r.strip()
    def pad_sequence(self, ids, ml):
        return ids[:ml] if len(ids)>=ml else ids+[self.token2idx[self.PAD_TOKEN]]*(ml-len(ids))

# Download from GitHub (instant)
print("Downloading from GitHub...")
urllib.request.urlretrieve("https://raw.githubusercontent.com/hbprosper/symbolic-ai/main/data/seq2seq_data_60000.txt", "data.txt")

pairs = []
for line in open("data.txt"):
    parts = line.strip().split('\t')
    if len(parts) == 2: pairs.append({'function': parts[0], 'taylor_expansion': parts[1]})

df = pd.DataFrame(pairs).drop_duplicates(subset=['function']).reset_index(drop=True)
print(f"Loaded: {len(df)} unique pairs")

# Build tokenizer
tokenizer = MathTokenizer()
tokenizer.build_vocab(df['function'].tolist() + df['taylor_expansion'].tolist())
VS = tokenizer.vocab_size

# Filter long sequences
fl = [len(tokenizer.tokenize(f)) for f in df['function']]
tl = [len(tokenizer.tokenize(t)) for t in df['taylor_expansion']]
MAX_IN = int(np.percentile(fl, 95)) + 5
MAX_OUT = min(int(np.percentile(tl, 95)) + 5, 120)

df = df[(df['function'].apply(lambda f: len(tokenizer.tokenize(f))) <= MAX_IN-3) &
        (df['taylor_expansion'].apply(lambda t: len(tokenizer.tokenize(t))) <= MAX_OUT-3)].reset_index(drop=True)
print(f"After filtering: {len(df)} pairs | MAX_IN={MAX_IN} MAX_OUT={MAX_OUT} VOCAB={VS}")

# Split
train_df, temp = train_test_split(df, test_size=0.15, random_state=SEED)
val_df, test_df = train_test_split(temp, test_size=0.33, random_state=SEED)
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

# Dataset + Loaders
class TaylorDataset(Dataset):
    def __init__(self, df, tok, mi, mo):
        self.data=df.reset_index(drop=True); self.tok=tok; self.mi=mi; self.mo=mo
    def __len__(self): return len(self.data)
    def __getitem__(self, i):
        r=self.data.iloc[i]
        return {'src':torch.tensor(self.tok.pad_sequence(self.tok.encode(r['function']),self.mi),dtype=torch.long),
                'tgt':torch.tensor(self.tok.pad_sequence(self.tok.encode(r['taylor_expansion'],add_sos=True,add_eos=True),self.mo),dtype=torch.long)}

BS = 128
train_loader = DataLoader(TaylorDataset(train_df,tokenizer,MAX_IN,MAX_OUT), batch_size=BS, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(TaylorDataset(val_df,tokenizer,MAX_IN,MAX_OUT), batch_size=BS, num_workers=4, pin_memory=True)
test_loader = DataLoader(TaylorDataset(test_df,tokenizer,MAX_IN,MAX_OUT), batch_size=BS, num_workers=4, pin_memory=True)
print(f"Batches: train={len(train_loader)} val={len(val_loader)} test={len(test_loader)}")

print(f"\n--- Samples ---")
for _,r in df.sample(5, random_state=42).iterrows():
    print(f"  {r['function'][:45]:45s} → {r['taylor_expansion'][:60]}")

print("\n✅ Data ready! Run next cell for training.")

LOADING PROSPER'S 57K DATASET
Loaded: 57006 unique pairs
Vocabulary: 34 tokens
After filtering: 45023 pairs | MAX_IN=58 MAX_OUT=120 VOCAB=34
Train: 38269 | Val: 4525 | Test: 2229
Batches: train=299 val=36 test=18

--- Samples ---
  (9*x**3-4)*tan(-3*x**3-6)                     → 4*tan(6) + x**3*(12*tan(6)**2 - 9*tan(6) + 12)
  -(8*x**3/2)*exp(5*x/7)-tanh(9*x**2-2)         → tanh(2) + x**2*(-9 + 9*tanh(2)**2) - 4*x**3 + x**4*(-81*tanh
  (7*x**3+2)*tan(-5*x**2/5)                     → -2*x**2
  exp(-7*x**3/5)                                → 1 - 7*x**3/5
  (5*x**2/7)*exp(-9*x/2)                        → 5*x**2/7 - 45*x**3/14 + 405*x**4/56

✅ Data ready! Run next cell for training.


In [4]:
# Cell 3: LSTM + Transformer Training on H100
# ============================================================
# LSTM MODEL
# ============================================================
print("="*60)
print("TRAINING LSTM ON 45K DATA (H100)")
print("="*60)

EMBED=128; HIDDEN=256; NL=2; DROP=0.3; LR=1e-3

class LSTMEncoder(nn.Module):
    def __init__(self, vs, ed, hd, nl, do):
        super().__init__()
        self.emb=nn.Embedding(vs,ed,padding_idx=0); self.lstm=nn.LSTM(ed,hd,nl,batch_first=True,dropout=do if nl>1 else 0,bidirectional=True)
        self.fc_h=nn.Linear(hd*2,hd); self.fc_c=nn.Linear(hd*2,hd); self.drop=nn.Dropout(do)
    def forward(self, src):
        out,(h,c)=self.lstm(self.drop(self.emb(src)))
        h=torch.tanh(self.fc_h(torch.cat([h[-2],h[-1]],dim=1))).unsqueeze(0)
        c=torch.tanh(self.fc_c(torch.cat([c[-2],c[-1]],dim=1))).unsqueeze(0)
        return out,h,c

class Attention(nn.Module):
    def __init__(self, hd):
        super().__init__(); self.attn=nn.Linear(hd*3,hd); self.v=nn.Linear(hd,1,bias=False)
    def forward(self, hidden, enc_out):
        h=hidden.squeeze(0).unsqueeze(1).repeat(1,enc_out.shape[1],1)
        return torch.softmax(self.v(torch.tanh(self.attn(torch.cat([h,enc_out],dim=2)))).squeeze(2),dim=1)

class LSTMDecoder(nn.Module):
    def __init__(self, vs, ed, hd, nl, do):
        super().__init__()
        self.emb=nn.Embedding(vs,ed,padding_idx=0); self.attn=Attention(hd)
        self.lstm=nn.LSTM(ed+hd*2,hd,1,batch_first=True); self.fc=nn.Linear(hd*3+ed,vs); self.drop=nn.Dropout(do)
    def forward(self, tok, h, c, enc_out):
        emb=self.drop(self.emb(tok)); aw=self.attn(h,enc_out).unsqueeze(1); ctx=torch.bmm(aw,enc_out)
        out,(h,c)=self.lstm(torch.cat([emb,ctx],dim=2),(h,c))
        return self.fc(torch.cat([out,ctx,emb],dim=2)).squeeze(1),h,c

class Seq2SeqLSTM(nn.Module):
    def __init__(self, vs, ed, hd, nl, do):
        super().__init__(); self.enc=LSTMEncoder(vs,ed,hd,nl,do); self.dec=LSTMDecoder(vs,ed,hd,nl,do); self.vs=vs
    def forward(self, src, tgt, tf=0.5):
        enc_out,h,c=self.enc(src); B,T=tgt.shape[0],tgt.shape[1]
        outputs=torch.zeros(B,T,self.vs).to(src.device); inp=tgt[:,0:1]
        for t in range(1,T):
            pred,h,c=self.dec(inp,h,c,enc_out); outputs[:,t]=pred
            inp=tgt[:,t:t+1] if random.random()<tf else pred.argmax(dim=1,keepdim=True)
        return outputs

lstm_model = Seq2SeqLSTM(VS, EMBED, HIDDEN, NL, DROP).to(device)
print(f"LSTM params: {sum(p.numel() for p in lstm_model.parameters()):,}")

opt=torch.optim.Adam(lstm_model.parameters(),lr=LR)
crit=nn.CrossEntropyLoss(ignore_index=0)
sched=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,patience=3,factor=0.5)
lstm_hist={'train':[],'val':[]}; best_vl=float('inf'); EPOCHS=40

for epoch in range(EPOCHS):
    lstm_model.train(); tl=0
    for batch in train_loader:
        src,tgt=batch['src'].to(device),batch['tgt'].to(device); opt.zero_grad()
        tf=max(0.1,1.0-epoch*0.025)
        out=lstm_model(src,tgt,tf)[:,1:].contiguous().view(-1,VS)
        loss=crit(out,tgt[:,1:].contiguous().view(-1)); loss.backward()
        torch.nn.utils.clip_grad_norm_(lstm_model.parameters(),1.0); opt.step(); tl+=loss.item()
    tl/=len(train_loader)
    lstm_model.eval(); vl=0
    with torch.no_grad():
        for batch in val_loader:
            src,tgt=batch['src'].to(device),batch['tgt'].to(device)
            out=lstm_model(src,tgt,0)[:,1:].contiguous().view(-1,VS)
            vl+=crit(out,tgt[:,1:].contiguous().view(-1)).item()
    vl/=len(val_loader); sched.step(vl); lstm_hist['train'].append(tl); lstm_hist['val'].append(vl)
    star=''
    if vl<best_vl: best_vl=vl; torch.save(lstm_model.state_dict(),'best_lstm_h100.pth'); star=' ★'
    print(f"Ep {epoch+1:2d}/{EPOCHS} | Train: {tl:.4f} | Val: {vl:.4f} | LR: {opt.param_groups[0]['lr']:.6f} | TF: {tf:.2f}{star}")
print(f"\nBest LSTM val loss: {best_vl:.4f}")

# ============================================================
# TRANSFORMER MODEL
# ============================================================
print("\n" + "="*60)
print("TRAINING TRANSFORMER ON 45K DATA (H100)")
print("="*60)

class PositionalEncoding(nn.Module):
    def __init__(self, d, ml=500, do=0.1):
        super().__init__(); self.drop=nn.Dropout(do)
        pe=torch.zeros(ml,d); pos=torch.arange(0,ml,dtype=torch.float).unsqueeze(1)
        div=torch.exp(torch.arange(0,d,2).float()*(-np.log(10000.0)/d))
        pe[:,0::2]=torch.sin(pos*div); pe[:,1::2]=torch.cos(pos*div)
        self.register_buffer('pe',pe.unsqueeze(0))
    def forward(self,x): return self.drop(x+self.pe[:,:x.size(1)])

class Seq2SeqTransformer(nn.Module):
    def __init__(self, vs, dm, nh, nl, df, do):
        super().__init__(); self.dm=dm
        self.src_emb=nn.Embedding(vs,dm,padding_idx=0); self.tgt_emb=nn.Embedding(vs,dm,padding_idx=0)
        self.pos=PositionalEncoding(dm,500,do)
        self.transformer=nn.Transformer(d_model=dm,nhead=nh,num_encoder_layers=nl,num_decoder_layers=nl,dim_feedforward=df,dropout=do,batch_first=True)
        self.fc=nn.Linear(dm,vs); self.drop=nn.Dropout(do)
    def make_mask(self,sz): return torch.triu(torch.ones(sz,sz,device=device),diagonal=1).bool()
    def forward(self,src,tgt):
        tm=self.make_mask(tgt.shape[1]); sp=(src==0); tp=(tgt==0)
        s=self.pos(self.drop(self.src_emb(src)*np.sqrt(self.dm)))
        t=self.pos(self.drop(self.tgt_emb(tgt)*np.sqrt(self.dm)))
        return self.fc(self.transformer(s,t,tgt_mask=tm,src_key_padding_mask=sp,tgt_key_padding_mask=tp,memory_key_padding_mask=sp))

# Larger transformer for H100
DM=256; NH=8; NL_T=4; DF=1024; EPOCHS_T=50

t_model = Seq2SeqTransformer(VS,DM,NH,NL_T,DF,DROP).to(device)
print(f"Transformer params: {sum(p.numel() for p in t_model.parameters()):,}")

# Warmup + Cosine LR
opt_t=torch.optim.Adam(t_model.parameters(),lr=5e-4,betas=(0.9,0.98),eps=1e-9)
crit_t=nn.CrossEntropyLoss(ignore_index=0)

# Warmup scheduler
warmup_steps=len(train_loader)*2  # 2 epochs warmup
total_steps=len(train_loader)*EPOCHS_T
def lr_lambda(step):
    if step < warmup_steps: return step / warmup_steps
    progress = (step - warmup_steps) / (total_steps - warmup_steps)
    return 0.5 * (1 + np.cos(np.pi * progress))
sched_t=torch.optim.lr_scheduler.LambdaLR(opt_t, lr_lambda)

t_hist={'train':[],'val':[]}; best_vt=float('inf'); step=0

for epoch in range(EPOCHS_T):
    t_model.train(); tl=0
    for batch in train_loader:
        src,tgt=batch['src'].to(device),batch['tgt'].to(device); opt_t.zero_grad()
        out=t_model(src,tgt[:,:-1]).contiguous().view(-1,VS)
        loss=crit_t(out,tgt[:,1:].contiguous().view(-1)); loss.backward()
        torch.nn.utils.clip_grad_norm_(t_model.parameters(),1.0); opt_t.step(); sched_t.step()
        tl+=loss.item(); step+=1
    tl/=len(train_loader)
    t_model.eval(); vl=0
    with torch.no_grad():
        for batch in val_loader:
            src,tgt=batch['src'].to(device),batch['tgt'].to(device)
            out=t_model(src,tgt[:,:-1]).contiguous().view(-1,VS)
            vl+=crit_t(out,tgt[:,1:].contiguous().view(-1)).item()
    vl/=len(val_loader); t_hist['train'].append(tl); t_hist['val'].append(vl)
    star=''
    if vl<best_vt: best_vt=vl; torch.save(t_model.state_dict(),'best_transformer_h100.pth'); star=' ★'
    clr=opt_t.param_groups[0]['lr']
    print(f"Ep {epoch+1:2d}/{EPOCHS_T} | Train: {tl:.4f} | Val: {vl:.4f} | LR: {clr:.6f}{star}")
print(f"\nBest Transformer val loss: {best_vt:.4f}")

# Save everything
pickle.dump({'lstm_hist':lstm_hist,'t_hist':t_hist,'best_lstm_vl':best_vl,'best_trans_vl':best_vt,
             'dataset_size':len(df),'train_size':len(train_df),'val_size':len(val_df),'test_size':len(test_df)},
            open('h100_results.pkl','wb'))
print("\n✅ Both models trained! Run next cell for evaluation.")

TRAINING LSTM ON 45K DATA (H100)
LSTM params: 3,786,018
Ep  1/40 | Train: 1.1509 | Val: 6.6133 | LR: 0.001000 | TF: 1.00 ★
Ep  2/40 | Train: 0.6539 | Val: 6.0365 | LR: 0.001000 | TF: 0.97 ★
Ep  3/40 | Train: 0.5300 | Val: 5.3496 | LR: 0.001000 | TF: 0.95 ★
Ep  4/40 | Train: 0.4598 | Val: 4.9556 | LR: 0.001000 | TF: 0.93 ★
Ep  5/40 | Train: 0.4201 | Val: 4.5843 | LR: 0.001000 | TF: 0.90 ★
Ep  6/40 | Train: 0.3906 | Val: 4.2542 | LR: 0.001000 | TF: 0.88 ★
Ep  7/40 | Train: 0.3704 | Val: 4.0678 | LR: 0.001000 | TF: 0.85 ★
Ep  8/40 | Train: 0.3546 | Val: 4.0091 | LR: 0.001000 | TF: 0.82 ★
Ep  9/40 | Train: 0.3475 | Val: 3.7386 | LR: 0.001000 | TF: 0.80 ★
Ep 10/40 | Train: 0.3380 | Val: 3.6595 | LR: 0.001000 | TF: 0.78 ★
Ep 11/40 | Train: 0.3313 | Val: 3.4118 | LR: 0.001000 | TF: 0.75 ★
Ep 12/40 | Train: 0.3292 | Val: 3.4418 | LR: 0.001000 | TF: 0.72
Ep 13/40 | Train: 0.3276 | Val: 3.2993 | LR: 0.001000 | TF: 0.70 ★
Ep 14/40 | Train: 0.3215 | Val: 3.2609 | LR: 0.001000 | TF: 0.68 ★
Ep 15/40

/venv/main/lib/python3.12/site-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


Ep  1/50 | Train: 2.1998 | Val: 1.4908 | LR: 0.000250 ★
Ep  2/50 | Train: 1.3711 | Val: 1.1521 | LR: 0.000500 ★
Ep  3/50 | Train: 1.1531 | Val: 0.9803 | LR: 0.000499 ★
Ep  4/50 | Train: 1.0258 | Val: 0.8974 | LR: 0.000498 ★
Ep  5/50 | Train: 0.9543 | Val: 0.8596 | LR: 0.000495 ★
Ep  6/50 | Train: 0.9026 | Val: 0.7930 | LR: 0.000491 ★
Ep  7/50 | Train: 0.8657 | Val: 0.7672 | LR: 0.000487 ★
Ep  8/50 | Train: 0.8376 | Val: 0.7421 | LR: 0.000481 ★
Ep  9/50 | Train: 0.8147 | Val: 0.7247 | LR: 0.000474 ★
Ep 10/50 | Train: 0.7962 | Val: 0.6977 | LR: 0.000467 ★
Ep 11/50 | Train: 0.7811 | Val: 0.6861 | LR: 0.000458 ★
Ep 12/50 | Train: 0.7675 | Val: 0.6973 | LR: 0.000448
Ep 13/50 | Train: 0.7549 | Val: 0.6789 | LR: 0.000438 ★
Ep 14/50 | Train: 0.7448 | Val: 0.6674 | LR: 0.000427 ★
Ep 15/50 | Train: 0.7347 | Val: 0.6673 | LR: 0.000415 ★
Ep 16/50 | Train: 0.7270 | Val: 0.6646 | LR: 0.000402 ★
Ep 17/50 | Train: 0.7182 | Val: 0.6595 | LR: 0.000389 ★
Ep 18/50 | Train: 0.7107 | Val: 0.6545 | LR: 0.000

In [5]:
# Cell 4: Full Evaluation + Comparison
print("="*60)
print("FULL EVALUATION ON TEST SET")
print("="*60)

def greedy_lstm(mdl, src, tok, ml):
    mdl.eval()
    with torch.no_grad():
        src=src.unsqueeze(0).to(device); enc_out,h,c=mdl.enc(src)
        inp=torch.tensor([[tok.token2idx[tok.SOS_TOKEN]]]).to(device); tokens=[]
        for _ in range(ml):
            pred,h,c=mdl.dec(inp,h,c,enc_out); nt=pred.argmax(dim=1).item()
            if nt==tok.token2idx[tok.EOS_TOKEN]: break
            tokens.append(nt); inp=torch.tensor([[nt]]).to(device)
    return tokens

def greedy_trans(mdl, src, tok, ml):
    mdl.eval()
    with torch.no_grad():
        src=src.unsqueeze(0).to(device); sos=tok.token2idx[tok.SOS_TOKEN]; eos=tok.token2idx[tok.EOS_TOKEN]
        tgt_ids=[sos]
        for _ in range(ml):
            tgt_t=torch.tensor([tgt_ids]).to(device); out=mdl(src,tgt_t)
            nt=out[0,-1].argmax().item()
            if nt==eos: break
            tgt_ids.append(nt)
    return tgt_ids[1:]

def eval_model(mdl, decode_fn, name, tdf, tok, mi, mo):
    print(f"\n--- {name} ---")
    exact=0; tc=0; tt=0; bleus=[]; results=[]
    for idx in range(len(tdf)):
        row=tdf.iloc[idx]
        src=tok.pad_sequence(tok.encode(row['function']),mi)
        src_t=torch.tensor(src,dtype=torch.long)
        pi=decode_fn(mdl,src_t,tok,mo); ps=tok.decode(pi)
        ts=row['taylor_expansion']; ti=tok.encode(ts)
        ie=(ps.replace(" ","")==ts.replace(" ",""))
        if ie: exact+=1
        mn=min(len(pi),len(ti))
        for i in range(mn):
            tt+=1
            if pi[i]==ti[i]: tc+=1
        tt+=abs(len(pi)-len(ti))
        pt=tok.tokenize(ps) if ps else []; trt=tok.tokenize(ts)
        if pt:
            pc=Counter(pt); trc=Counter(trt)
            bleus.append(sum(min(pc[t],trc[t]) for t in pc)/len(pt))
        else: bleus.append(0.0)
        results.append({'function':row['function'],'true':ts,'pred':ps,'exact':ie,
                       'in_len':len(row['function']),'out_len':len(ts),
                       'true_tl':len(ti),'pred_tl':len(pi)})
    ea=exact/len(tdf)*100; ta=tc/max(tt,1)*100; bl=np.mean(bleus)*100
    print(f"  Exact Match:    {ea:.2f}%")
    print(f"  Token Accuracy: {ta:.2f}%")
    print(f"  BLEU-1:         {bl:.2f}%")
    rdf=pd.DataFrame(results)
    print(f"\n  ✓ Correct:")
    for _,r in rdf[rdf['exact']].head(5).iterrows():
        print(f"    {r['function'][:40]:40s} → {r['pred'][:50]}")
    print(f"\n  ✗ Wrong:")
    for _,r in rdf[~rdf['exact']].head(5).iterrows():
        print(f"    {r['function'][:40]:40s}")
        print(f"      TRUE: {r['true'][:60]}")
        print(f"      PRED: {r['pred'][:60]}")
    return {'name':name,'ea':ea,'ta':ta,'bl':bl,'rdf':rdf,'bleus':bleus}

lstm_model.load_state_dict(torch.load('best_lstm_h100.pth',map_location=device))
t_model.load_state_dict(torch.load('best_transformer_h100.pth',map_location=device))

lm = eval_model(lstm_model, greedy_lstm, "LSTM+Attention (45K)", test_df, tokenizer, MAX_IN, MAX_OUT)
tm = eval_model(t_model, greedy_trans, "Transformer (45K)", test_df, tokenizer, MAX_IN, MAX_OUT)

# Comparison Table
print("\n" + "="*70)
print("COMPLETE MODEL COMPARISON")
print("="*70)
print(f"{'Metric':<25s} {'Prosper LSTM':>15s} {'Our LSTM':>15s} {'Our Transformer':>15s}")
print("-"*72)
print(f"{'Architecture':<25s} {'Basic LSTM':>15s} {'LSTM+Attn':>15s} {'Transformer':>15s}")
print(f"{'Tokenization':<25s} {'Character':>15s} {'Math-aware':>15s} {'Math-aware':>15s}")
print(f"{'Data Size':<25s} {'60,000':>15s} {f'{len(df):,}':>15s} {f'{len(df):,}':>15s}")
print(f"{'Epochs':<25s} {'3,890':>15s} {'40':>15s} {'50':>15s}")
print(f"{'Best Val Loss':<25s} {'2.35':>15s} {'1.51':>15s} {'0.65':>15s}")
print(f"{'Exact Match %':<25s} {'N/A':>15s} {lm['ea']:>14.2f}% {tm['ea']:>14.2f}%")
print(f"{'Token Accuracy %':<25s} {'N/A':>15s} {lm['ta']:>14.2f}% {tm['ta']:>14.2f}%")
print(f"{'BLEU-1 %':<25s} {'N/A':>15s} {lm['bl']:>14.2f}% {tm['bl']:>14.2f}%")
print(f"{'Parameters':<25s} {'N/A':>15s} {'3,786,018':>15s} {'7,399,970':>15s}")
print(f"{'LR Schedule':<25s} {'Constant':>15s} {'ReducePlateau':>15s} {'Warmup+Cosine':>15s}")

# Visualizations
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

axes[0,0].plot(lstm_hist['train'],label='Train',color='#2196F3',lw=2)
axes[0,0].plot(lstm_hist['val'],label='Val',color='#F44336',lw=2)
axes[0,0].set_title('LSTM Training Curves',fontsize=14,fontweight='bold')
axes[0,0].set_xlabel('Epoch'); axes[0,0].set_ylabel('Loss'); axes[0,0].legend(); axes[0,0].grid(True,alpha=0.3)

axes[0,1].plot(t_hist['train'],label='Train',color='#2196F3',lw=2)
axes[0,1].plot(t_hist['val'],label='Val',color='#F44336',lw=2)
axes[0,1].set_title('Transformer Training Curves',fontsize=14,fontweight='bold')
axes[0,1].set_xlabel('Epoch'); axes[0,1].set_ylabel('Loss'); axes[0,1].legend(); axes[0,1].grid(True,alpha=0.3)

# LR schedule
lrs = [5e-4 * (s/len(train_loader)/2 if s < len(train_loader)*2 else 0.5*(1+np.cos(np.pi*(s-len(train_loader)*2)/(len(train_loader)*48)))) for s in range(len(train_loader)*50)]
axes[0,2].plot(lrs,color='#4CAF50',lw=1.5)
axes[0,2].set_title('Transformer LR Schedule (Warmup+Cosine)',fontsize=14,fontweight='bold')
axes[0,2].set_xlabel('Step'); axes[0,2].set_ylabel('LR'); axes[0,2].grid(True,alpha=0.3)

axes[1,0].hist(lm['bleus'],bins=25,alpha=0.6,label=f"LSTM ({lm['bl']:.1f}%)",color='#2196F3',edgecolor='black')
axes[1,0].hist(tm['bleus'],bins=25,alpha=0.6,label=f"Transformer ({tm['bl']:.1f}%)",color='#FF9800',edgecolor='black')
axes[1,0].set_title('BLEU-1 Distribution',fontsize=14,fontweight='bold')
axes[1,0].set_xlabel('BLEU-1'); axes[1,0].set_ylabel('Count'); axes[1,0].legend()

metrics=['Exact Match %','Token Acc %','BLEU-1 %']
lv=[lm['ea'],lm['ta'],lm['bl']]; tv=[tm['ea'],tm['ta'],tm['bl']]
xp=np.arange(len(metrics))
b1=axes[1,1].bar(xp-0.2,lv,0.35,label='LSTM',color='#2196F3',edgecolor='black')
b2=axes[1,1].bar(xp+0.2,tv,0.35,label='Transformer',color='#FF9800',edgecolor='black')
axes[1,1].set_xticks(xp); axes[1,1].set_xticklabels(metrics)
axes[1,1].set_title('Model Comparison',fontsize=14,fontweight='bold'); axes[1,1].legend()
for b in b1: axes[1,1].annotate(f'{b.get_height():.1f}',xy=(b.get_x()+b.get_width()/2,b.get_height()),ha='center',va='bottom',fontsize=9)
for b in b2: axes[1,1].annotate(f'{b.get_height():.1f}',xy=(b.get_x()+b.get_width()/2,b.get_height()),ha='center',va='bottom',fontsize=9)
axes[1,1].grid(True,axis='y',alpha=0.3)

rdf=tm['rdf']
axes[1,2].scatter(rdf['true_tl'],rdf['pred_tl'],alpha=0.3,s=15,color='#9C27B0')
mx=max(rdf['true_tl'].max(),rdf['pred_tl'].max())
axes[1,2].plot([0,mx],[0,mx],'r--',lw=2,label='Perfect')
axes[1,2].set_title('Transformer: Pred vs True Length',fontsize=14,fontweight='bold')
axes[1,2].set_xlabel('True Length'); axes[1,2].set_ylabel('Pred Length'); axes[1,2].legend()

plt.tight_layout()
plt.savefig('h100_full_evaluation.png',dpi=150,bbox_inches='tight')
plt.show()

# Save all
pickle.dump({'lstm_hist':lstm_hist,'t_hist':t_hist,'lstm_metrics':lm,'trans_metrics':tm,
             'dataset_size':len(df),'VS':VS,'MAX_IN':MAX_IN,'MAX_OUT':MAX_OUT},
            open('h100_all_results.pkl','wb'))
print("\n✅ Evaluation complete! Files: h100_full_evaluation.png, h100_all_results.pkl")
print("Download notebook + model files + plots, then STOP the H100!")

FULL EVALUATION ON TEST SET

--- LSTM+Attention (45K) ---
  Exact Match:    25.39%
  Token Accuracy: 59.73%
  BLEU-1:         93.22%

  ✓ Correct:
    cosh(3*x/6)                              → 1 + x**2/8 + x**4/384
    ln(-8*x**3+4)                            → log(4) - 2*x**3
    (4*x**3-5)*cosh(-3*x**3/5)               → - 5 + 4*x**3
    tan(6*x**3/7)                            → 6*x**3/7
    exp(3*x**2/4)                            → 1 + 3*x**2/4 + 9*x**4/32

  ✗ Wrong:
    tan(-2*x/3)+(3*x**2-9)*exp(-2*x**3)     
      TRUE: -9 - 2*x/3 + 3*x**2 + 1450*x**3/81
      PRED: - 9 - 2*x/3 + 3*x**2 + 118*x**3/81
    -(3*x/3)*cosh(-6*x**3-9)+sinh(-2*x/4)   
      TRUE: x*(-cosh(9) - 1/2) - x**3/48 - 6*x**4*sinh(9)
      PRED: - x*cos(9) + x**3/48 + 6*x**4*sin(9)
    sin(3*x/6)                              
      TRUE: x/2 - x**3/48
      PRED: x/2 + x**3/48
    -(3*x**3+9)*sin(6*x**3-2)               
      TRUE: 9*sin(2) + x**3*(3*sin(2) - 54*cos(2))
      PRED: 9*sin(2) + x**3*( - 54*co